# GPU Pulse · FLOP/s 검증

**Colab 런타임에서 NVIDIA L4를 먼저 선택하고, 가능하면 A100에서도 다시 실행하세요.**  
런타임 → 런타임 유형 변경 → GPU → **L4** 또는 **A100** → 모두 실행. GPU 선택지는 계정과 런타임 할당 상황에 따라 달라집니다. [Colab FAQ](https://research.google.com/colaboratory/faq.html)

| 할당 GPU | 실행 범위 |
| --- | --- |
| **L4** | FP16·BF16·TF32 Tensor, FP32 CUDA, FP8 Tensor 시도. 실제 카운터 및 커널 지원에 따라 개별 판정 |
| **A100** | FP16·BF16·TF32 Tensor, FP32 CUDA. FP8은 건너뜀. GA100용 SYSCLK 샘플링 사용 |
| **T4 또는 다른 GPU** | CPU 산술 회귀 테스트만 실행. T4는 현재 GPU Pulse PM Sampling 설정의 검증 대상에서 제외 |

이 노트북은 `main` 기준 커밋 `2a1521e`의 실제 계산 코드를 불러옵니다. **PASS**는 해당 GPU와 카운터에서 현재 계산식이 선택된 오차 기준 안에 들어온다는 뜻입니다. **FAIL**은 재현된 수치 불일치, **SKIP**은 지원 경로 없음, **INCONCLUSIVE**는 권한·샘플 품질·실행 조건 때문에 판정할 수 없음을 뜻합니다. CPU PASS는 GPU 실측 PASS와 별개입니다.

Colab에서는 운영 노드의 DCGM 데몬 상태를 재현할 수 없습니다. GPU Pulse의 Tensor provisional 상태나 운영 설정은 이 테스트로 변경되지 않습니다. [CUPTI PM Sampling](https://docs.nvidia.com/cupti-python/api-reference/topics/pm_sampling.html)

코드 셀 하나를 실행하세요. 카탈로그에 없는 정확한 카운터도 CUPTI에 직접 요청합니다. GEMM 단독 결과와 카운터 검증 결과가 별도로 표시됩니다.


In [ ]:
import json
import math
import os
import sys
import tempfile
import urllib.request
from importlib import metadata
from pathlib import Path

# Keep the production baseline immutable even if main changes later.
BASE = "2a1521e3e37f518671bc7454d79eab7ec95c8535"
ROOT = Path(tempfile.mkdtemp(prefix="gpu-pulse-baseline-"))
for name in ("metrics.py", "sample_schema.py", "sample_history.py", "collector_config.py",
             "gpu_discovery.py", "observer.py", "peak_registry.py", "peak_specs.json"):
    url = f"https://raw.githubusercontent.com/myeongsu-koo/gpu-pulse/{BASE}/main/{name}"
    with urllib.request.urlopen(url, timeout=30) as response:
        (ROOT / name).write_bytes(response.read())
sys.path.insert(0, str(ROOT))

# Use the CUDA Runtime availability check in PyTorch.
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "0"

from metrics import KEY_TO_SPEC, precision_dashboard_values, to_dashboard_values
from peak_registry import compare_compute

def cpu_checks():
    keys = ("fadd", "fmul", "ffma", "tensor_fp16")
    specs = [KEY_TO_SPEC[key] for key in keys]
    raw = {spec.name: value for spec, value in zip(specs, (1e12, 2e12, 3e12, 4e12), strict=True)}
    for interval in (0.01, 0.1):
        output = to_dashboard_values(raw, specs, interval)
        assert output["cuda_fp32_tflops"] == 9, output
        assert output["tensor_tflops"] == 4, output
    precision = precision_dashboard_values(raw, specs)
    row = precision["fp16.tensor.dense"]
    assert row["value"] == 4 and row["counter_unit_verified"] is False
    assert compare_compute("fp16.tensor.dense", row, {"value": 8, "unit": "TFLOP/s"})["attainment_pct"] is None
    del raw[specs[1].name]
    assert to_dashboard_values(raw, specs)["cuda_fp32_tflops"] is None

cpu_checks()
print(f"PASS · CPU 산술 회귀 테스트 · GPU Pulse {BASE[:7]}")


import importlib.util
import subprocess
import time
from statistics import median

import torch
from observer import CudaPrimaryContext, discover_sets, get_chip_name

SIZE = 4096
TRIALS = 3
TARGET_SECONDS = 2.0
RATE_TOLERANCE = 0.10
MIN_COVERAGE = 0.95
MAX_COVERAGE = 1.05
MAX_HOST_SLACK = 0.05

CASES = {
    "FP32 CUDA": (("fadd", "fmul", "ffma"), "fp32.cuda.dense", "cuda_fp32_tflops"),
    "FP16 Tensor": (("tensor_fp16",), "fp16.tensor.dense", "tensor_tflops"),
    "BF16 Tensor": (("tensor_bf16",), "bf16.tensor.dense", "tensor_tflops"),
    "TF32 Tensor": (("tensor_tf32",), "tf32.tensor.dense", "tensor_tflops"),
    "FP8 Tensor": (("tensor_fp8",), "fp8.tensor.dense", "tensor_tflops"),
}

def make_gemm(case):
    if case == "FP8 Tensor":
        if not hasattr(torch, "_scaled_mm") or not hasattr(torch, "float8_e4m3fn"):
            raise RuntimeError("PyTorch FP8 scaled GEMM API unavailable")
        a = torch.randn((SIZE, SIZE), device="cuda").mul_(0.1).to(torch.float8_e4m3fn)
        b = torch.randn((SIZE, SIZE), device="cuda").mul_(0.1).to(torch.float8_e4m3fn).t()
        scale = torch.ones((), device="cuda", dtype=torch.float32)
        return lambda: torch._scaled_mm(a, b, scale_a=scale, scale_b=scale, out_dtype=torch.float32)
    dtype = {"FP16 Tensor": torch.float16, "BF16 Tensor": torch.bfloat16}.get(case, torch.float32)
    a = torch.randn((SIZE, SIZE), dtype=dtype, device="cuda")
    b = torch.randn_like(a)
    out = torch.empty_like(a)
    return lambda: torch.mm(a, b, out=out)

def timed_gemms(work, iterations):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    start.record()
    for _ in range(iterations):
        work()
    end.record()
    end.synchronize()
    return start.elapsed_time(end) / 1000

def select_collector(pm, sets, specs, chip):
    bases = {spec.base for spec in specs}
    names = [spec.name for spec in specs]
    candidates = [name for name, metrics in sorted(sets.items())
                  if bases.issubset({metric.split(".", 1)[0] for metric in metrics})]
    errors = []
    # The catalog is a preflight hint. Let the evaluator attempt these exact metrics.
    for name in [*candidates, None]:
        collector = pm.Collector(device_index=0)
        try:
            collector.enable()
        except Exception as error:
            raise RuntimeError(f"CUPTI enable failed: {error}") from error
        try:
            ga100 = chip.lower().startswith("ga100")
            collector.configure(
                metrics=names, hardware_buffer_size=128 * 1024 * 1024,
                sampling_interval=10_000_000 if ga100 else 100_000_000,
                trigger_mode=pm.TriggerMode.GPU_SYSCLK_INTERVAL if ga100 else pm.TriggerMode.GPU_TIME_INTERVAL,
                hw_buffer_append_mode=pm.HardwareBufferAppendMode.KEEP_LATEST,
                single_pass_metric_set_name=name,
            )
            return collector, name or "DIRECT (no named set)"
        except Exception as error:
            errors.append(f"{name or 'DIRECT'}: {error}")
            collector.disable()
    return None, "CUPTI could not configure exact counters: " + " | ".join(errors)

def capture(collector, work, count, specs, precision_key, card_key):
    torch.cuda.synchronize()
    host_start = time.perf_counter()
    collector.start()
    try:
        seconds = timed_gemms(work, count)
    finally:
        collector.stop()
    host_elapsed = time.perf_counter() - host_start
    # Integrate complete sample windows directly; never extrapolate missing samples.
    covered = 0.0
    total = 0.0
    samples = 0
    prev_end = -1
    first_start = None
    while True:
        batch = collector.decode(max_samples=4096)
        advanced = False
        for sample in batch:
            start, end = int(sample.start_timestamp), int(sample.end_timestamp)
            if end <= prev_end or end <= start:
                continue
            if start < prev_end:
                continue
            prev_end, advanced = end, True
            raw = dict(zip(batch.metrics, sample.metric_values, strict=True))
            duration = (end - start) / 1e9
            row = precision_dashboard_values(raw, specs).get(precision_key, {})
            table = row.get("value")
            card = to_dashboard_values(raw, specs, duration).get(card_key)
            if table is None or card is None:
                continue
            if not math.isclose(table, card, rel_tol=1e-12):
                raise AssertionError(f"Production card and precision table disagree: {card} vs {table}")
            first_start = start if first_start is None else first_start
            covered += duration
            total += card * duration
            samples += 1
        if not advanced:
            break
    expected = 2 * SIZE**3 * count / 1e12
    coverage = covered / seconds if seconds > 0 else 0
    host_slack = (host_elapsed - seconds) / seconds if seconds > 0 else None
    valid = (samples >= 10 and MIN_COVERAGE <= coverage <= MAX_COVERAGE
             and host_slack is not None and 0 <= host_slack <= MAX_HOST_SLACK)
    return {
        "gemms": count, "cuda_seconds": seconds, "host_seconds": host_elapsed,
        "samples": samples, "sample_seconds": covered, "coverage": coverage,
        "sample_span_seconds": (prev_end - first_start) / 1e9 if first_start is not None else None,
        "host_slack": host_slack, "eligible": valid,
        "gemm_tflops": expected / seconds if seconds > 0 else None,
        "gpu_pulse_tflops": total / covered if covered else None,
        "expected_over_gpu_pulse": expected / total if total > 0 else None,
    }

def verdict(trials):
    if not trials or not all(trial["eligible"] for trial in trials):
        return "INCONCLUSIVE", "Incomplete sampling or excessive host/CUDA timing difference"
    ratios = [trial["expected_over_gpu_pulse"] for trial in trials]
    if any(ratio is None for ratio in ratios):
        return "INCONCLUSIVE", "No positive matching counter; check the actual execution path"
    if all(abs(ratio - 1) <= RATE_TOLERANCE for ratio in ratios):
        return "PASS", "Current formula matches the known GEMM work within ±10%"
    if all(abs(ratio / 2 - 1) <= RATE_TOLERANCE for ratio in ratios):
        return "FAIL", "Consistent ≈2× difference; current rate is about half the GEMM count"
    if all(abs(ratio * 2 - 1) <= RATE_TOLERANCE for ratio in ratios):
        return "FAIL", "Consistent ≈0.5× difference; current rate is about twice the GEMM count"
    if (all(ratio > 1 + RATE_TOLERANCE for ratio in ratios)
            or all(ratio < 1 - RATE_TOLERANCE for ratio in ratios)):
        return "FAIL", "All eligible repeats disagree with the current formula"
    return "INCONCLUSIVE", "Repeats disagree on whether the formula matches"

def test_case(case, config, pm, sets, chip):
    keys, precision_key, card_key = config
    specs = [KEY_TO_SPEC[key] for key in keys]
    baseline = {}
    backend = torch.backends.cuda.matmul
    old = backend.allow_tf32, backend.allow_fp16_reduced_precision_reduction, backend.allow_bf16_reduced_precision_reduction
    try:
        # A minimum FP32 matmul precision setting alone doesn't identify the selected kernel.
        backend.allow_tf32 = case == "TF32 Tensor"
        backend.allow_fp16_reduced_precision_reduction = False
        backend.allow_bf16_reduced_precision_reduction = False
        try:
            work = make_gemm(case)
            timed_gemms(work, 8)
        except (RuntimeError, TypeError, AttributeError) as error:
            return {"case": case, "status": "SKIP", "reason": f"GEMM workload unavailable: {error}"}
        pilot = timed_gemms(work, 32)
        if not math.isfinite(pilot) or pilot <= 0:
            return {"case": case, "status": "INCONCLUSIVE", "reason": "Invalid CUDA event pilot duration"}
        count = min(32768, max(96, round(32 * TARGET_SECONDS / pilot)))
        baseline = {
            "standalone_gemm_tflops": 2 * SIZE**3 * 32 / pilot / 1e12,
            "catalog_sets_by_counter": {
                spec.name: [name for name, metrics in sets.items()
                            if any(metric.split(".", 1)[0] == spec.base for metric in metrics)]
                for spec in specs
            },
        }
        collector, selected = select_collector(pm, sets, specs, chip)
        if collector is None:
            return {"case": case, "status": "INCONCLUSIVE", "reason": selected, **baseline}
        try:
            trials = [capture(collector, work, count, specs, precision_key, card_key)
                      for _ in range(TRIALS)]
        finally:
            collector.disable()
    except AssertionError as error:
        return {"case": case, "status": "FAIL", "reason": str(error)}
    except Exception as error:
        return {"case": case, "status": "INCONCLUSIVE", "reason": f"{type(error).__name__}: {error}", **baseline}
    finally:
        backend.allow_tf32, backend.allow_fp16_reduced_precision_reduction, backend.allow_bf16_reduced_precision_reduction = old
    status, reason = verdict(trials)
    return {"case": case, "status": status, "reason": reason, "single_pass_set": selected,
            "counters": [spec.name for spec in specs], "trials": trials, **baseline}


from IPython.display import display
import pandas as pd

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
family = next((model for model in ("L4", "A100") if model in gpu.upper().split("-")[0].split()), None)
report = {"gpu": gpu, "baseline": BASE, "cpu_sanity": "PASS", "torch": torch.__version__,
          "cuda": torch.version.cuda, "dcgm": "NOT_TESTED_MANAGED_COLAB", "results": []}
if family is None:
    report["results"] = [{"case": case, "status": "SKIP", "reason": "Select NVIDIA L4 or A100 in Colab"}
                         for case in CASES]
else:
    try:
        if importlib.util.find_spec("cupti") is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cupti-python>=13.2,<14"])
        from cupti import pm_sampling
        report["cupti_python"] = metadata.version("cupti-python")
        with CudaPrimaryContext(0):
            chip = get_chip_name(0)
            report["chip"] = chip
            try:
                sets = discover_sets(chip)
            except Exception as error:
                report["catalog_error"] = f"{type(error).__name__}: {error}"
                sets = {}
            report["catalog_sets"] = {name: sorted(metrics) for name, metrics in sets.items()}
            print(f"Chip: {chip} | CUPTI Python: {report['cupti_python']} | Catalog sets: {list(sets)}")
            for case, config in CASES.items():
                if family == "A100" and case == "FP8 Tensor":
                    result = {"case": case, "status": "SKIP", "reason": "A100 has no native FP8 Tensor path"}
                else:
                    result = test_case(case, config, pm_sampling, sets, chip)
                report["results"].append(result)
                print(f'{result["status"]:12} {case}: {result["reason"]}')
    except Exception as error:
        checked = {row["case"] for row in report["results"]}
        report["results"].extend({"case": case, "status": "INCONCLUSIVE",
                                  "reason": f"GPU setup: {type(error).__name__}: {error}"}
                                 for case in CASES if case not in checked)

def show_results(report):
    rows = []
    for result in report["results"]:
        row = {"Test": result["case"], "Result": result["status"], "Reason": result["reason"]}
        standalone = result.get("standalone_gemm_tflops")
        row["Standalone GEMM TFLOP/s"] = round(standalone, 3) if standalone is not None else None
        row["Catalog matches"] = ", ".join(
            f"{counter.split('.')[0]}: {', '.join(sets) or 'absent'}"
            for counter, sets in result.get("catalog_sets_by_counter", {}).items()
        )
        trials = result.get("trials", [])
        for field, label in (("gemm_tflops", "CUDA GEMM TFLOP/s"),
                             ("gpu_pulse_tflops", "GPU Pulse TFLOP/s"),
                             ("expected_over_gpu_pulse", "Expected / GPU Pulse"),
                             ("coverage", "Sample coverage")):
            values = [trial[field] for trial in trials if trial[field] is not None]
            row[label] = round(median(values), 3) if values else None
        rows.append(row)
    display(pd.DataFrame(rows))

statuses = {row["status"] for row in report["results"]}
overall = ("FAIL" if "FAIL" in statuses else
           "PASS · all cases" if statuses == {"PASS"} else
           "PARTIAL · some cases passed" if "PASS" in statuses else "NO HARDWARE VERDICT")
print(f"Hardware verdict: {overall} | GPU: {gpu} | baseline: {BASE[:7]}")
show_results(report)
Path("gpu_pulse_flops_validation.json").write_text(json.dumps(report, indent=2, allow_nan=False), encoding="utf-8")
print("Saved gpu_pulse_flops_validation.json in the Colab Files panel.")


### 결과 해석

- `Expected / GPU Pulse`는 **같은 실행 구간**에서 GEMM의 이론 연산량을 CUPTI의 구간 적분 연산량으로 나눈 값입니다. 모든 반복이 **0.90–1.10**이면 현재 계수와 일치합니다. 2 근처에 반복해서 모이면 별도 카운터 보정의 근거가 됩니다.
- 시간 커버리지가 **95–105%** 밖이거나 CUDA 이벤트 시간 대비 호스트 구간 차이가 **5%**를 넘으면 판정을 보류합니다. 양의 카운터 값이 없으면 커널 경로나 카운터 가용성을 확인해야 합니다. 이는 측정창과 커널 경로를 완전히 증명하는 검증은 아닙니다. 결론은 해당 GPU, 런타임, 커널, 카운터에 한정됩니다.
- 결과를 공유할 때 마지막 셀에서 생성된 `gpu_pulse_flops_validation.json`을 첨부하면 반복별 수치, 카운터 이름과 지원/판정 사유를 함께 확인할 수 있습니다.